# Analyze DAG files created by tree_to_dag and larch_merge

This notebook analyzes the individual DAGs created from randomized alignments and the merged DAG produced by larch-dagutil.

## Import Python modules

In [18]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(font_scale=1.0, style='ticks', palette='colorblind')

## Parse log files and extract statistics

Extract parsimony scores and tree counts from tree_to_dag and larch_merge log files for HA H1 and HA H7.

In [ ]:
# Automatically discover all segment-subtype combinations from logs directory
segment_dirs = [d for d in os.listdir('../logs/') if os.path.isdir(os.path.join('../logs/', d))]

# Store statistics for each DAG
dag_stats = []

for segment in segment_dirs:
    subtype_dirs = [d for d in os.listdir(f'../logs/{segment}/') if os.path.isdir(os.path.join(f'../logs/{segment}/', d))]
    for subtype in subtype_dirs:
        print(f"\nProcessing {segment} {subtype}...")
        
        # Get number of randomizations from directory
        randomized_dirs = [d for d in os.listdir(f'../logs/{segment}/{subtype}/')
                          if d.startswith('randomized_') and 
                          os.path.isdir(os.path.join(f'../logs/{segment}/{subtype}/', d))]
        n_randomizations = len(randomized_dirs)
        
        if n_randomizations == 0:
            print(f"  Warning: No randomized directories found, skipping...")
            continue
        
        # Load individual DAGs from randomized runs
        for n in range(n_randomizations):
            log_file = f'../logs/{segment}/{subtype}/randomized_{n}/tree_to_dag.log'
            
            if not os.path.exists(log_file):
                print(f"  Warning: {log_file} not found, skipping...")
                continue
            
            # Read log file
            with open(log_file, 'r') as f:
                log_content = f.read()
            
            # Parse statistics using regex
            min_parsimony_match = re.search(r'Min parsimony score in DAG:\s*(\d+)', log_content)
            max_parsimony_match = re.search(r'Max parsimony score in DAG:\s*(\d+)', log_content)
            tree_count_match = re.search(r'Total trees in DAG:\s*(\d+)', log_content)
            
            if not all([min_parsimony_match, max_parsimony_match, tree_count_match]):
                print(f"  Warning: Could not parse all statistics from {log_file}")
                continue
            
            min_parsimony = int(min_parsimony_match.group(1))
            max_parsimony = int(max_parsimony_match.group(1))
            tree_count = int(tree_count_match.group(1))
            
            dag_stats.append({
                'segment_subtype': f'{segment}_{subtype}',
                'dag_type': 'individual',
                'randomization': n,
                'min_parsimony': min_parsimony,
                'max_parsimony': max_parsimony,
            })
            
            print(f"  Randomization {n}: {min_parsimony} parsimony, {tree_count:,} trees")
        
        # Load merged DAG statistics from larch_merge log
        merged_log_file = f'../logs/{segment}/{subtype}/larch_merge.log'
        
        if os.path.exists(merged_log_file):
            # Read log file
            with open(merged_log_file, 'r') as f:
                log_content = f.read()
            
            # Parse statistics using regex
            parsimony_min_match = re.search(r'parsimony_min:\s*score:(\d+)', log_content)
            parsimony_max_match = re.search(r'parsimony_max:\s*score:(\d+)', log_content)
            tree_count_match = re.search(r'tree_count:\s*(\d+)', log_content)
            num_leaves_match = re.search(r'DAG leave\(without trimming\):\s*(\d+)', log_content)
            num_nodes_match = re.search(r'DAG nodes\(without trimming\):\s*(\d+)', log_content)
            
            if not all([parsimony_min_match, parsimony_max_match]):
                print(f"  Warning: Could not parse parsimony statistics from {merged_log_file}")
            else:
                min_parsimony = int(parsimony_min_match.group(1))
                max_parsimony = int(parsimony_max_match.group(1))
                
                # Parse tree count as string (too large for float/int)
                tree_count_str = tree_count_match.group(1) if tree_count_match else None
                
                # Calculate approximate order of magnitude for display
                tree_count_magnitude = None
                if tree_count_str:
                    tree_count_magnitude = len(tree_count_str) - 1  # approximate log10
                
                num_leaves = int(num_leaves_match.group(1)) if num_leaves_match else None
                num_nodes = int(num_nodes_match.group(1)) if num_nodes_match else None
                
                dag_stats.append({
                    'segment_subtype': f'{segment}_{subtype}',
                    'dag_type': 'merged',
                    'randomization': None,
                    'min_parsimony': min_parsimony,
                    'max_parsimony': max_parsimony,
                })
                
                if tree_count_magnitude:
                    print(f"  Merged DAG: {min_parsimony} parsimony, ~10^{tree_count_magnitude} trees")
                else:
                    print(f"  Merged DAG: {min_parsimony} parsimony")
        else:
            print(f"  Warning: {merged_log_file} not found")

# Create DataFrame
dag_stats_df = pd.DataFrame(dag_stats)
print(f"\nLoaded statistics for {len(dag_stats_df)} DAGs across {dag_stats_df['segment_subtype'].nunique()} segment-subtype combinations")

## Parsimony score improvement from merging DAGs

Bar plot showing the improvement in minimum parsimony score achieved by merging individual DAGs. Improvement is calculated as (best individual DAG score - merged DAG score).

In [ ]:
# Calculate parsimony improvement for each segment/subtype
improvements = []

for segment_subtype in dag_stats_df['segment_subtype'].unique():
    subset = dag_stats_df[dag_stats_df['segment_subtype'] == segment_subtype]
    individual = subset[subset['dag_type'] == 'individual']
    merged = subset[subset['dag_type'] == 'merged']
    
    if len(merged) > 0 and len(individual) > 0:
        best_individual = individual['min_parsimony'].min()
        merged_parsimony = merged['min_parsimony'].iloc[0]
        improvement = best_individual - merged_parsimony
        
        improvements.append({
            'segment_subtype': segment_subtype,
            'improvement': improvement,
            'best_individual': best_individual,
            'merged': merged_parsimony
        })

improvements_df = pd.DataFrame(improvements)

# Define custom order: HA segments first, then NA segments, then the rest
ha_segments = sorted([s for s in improvements_df['segment_subtype'] if s.startswith('HA_')])
na_segments = sorted([s for s in improvements_df['segment_subtype'] if s.startswith('NA_')])
other_segments = sorted([s for s in improvements_df['segment_subtype'] if not s.startswith('HA_') and not s.startswith('NA_')])
segment_order = ha_segments + na_segments + other_segments

# Reorder dataframe
improvements_df_ordered = improvements_df.set_index('segment_subtype').loc[segment_order].reset_index()

# Create bar plot
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(improvements_df_ordered))
ax.bar(x, improvements_df_ordered['improvement'])

ax.set_xlabel('Segment / Subtype')
ax.set_ylabel('Parsimony Improvement\n(Best Individual - Merged)')
ax.set_title('Parsimony Score Improvement from Merging DAGs')

# Set x-tick labels
ax.set_xticks(x)
ax.set_xticklabels(improvements_df_ordered['segment_subtype'], rotation=90)

# Add value labels on bars
for i, (idx, row) in enumerate(improvements_df_ordered.iterrows()):
    ax.text(i, row['improvement'] + max(improvements_df_ordered['improvement']) * 0.01, 
            f"{int(row['improvement'])}", 
            ha='center', va='bottom', fontsize=8)

# Add horizontal line at y=0
ax.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.3)

sns.despine(ax=ax)
ax.yaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()